# Setting I/O Locations

# Importing Date

In [1]:
!!pip install openpyxl


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 3, Finished, Available, Finished, False)

['Requirement already satisfied: openpyxl in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (3.0.10)',
 'Requirement already satisfied: et_xmlfile in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (from openpyxl) (1.1.0)']

In [2]:
# List all items in Lakehouse Files section

files = mssparkutils.fs.ls("Files")

for file in files:
    print(file.name)


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 4, Finished, Available, Finished, False)

Bill1_Schedule.xlsx
Bill2_Schedule.xlsx
CASH
Gold_Output
LST.xlsx
RO_Attempts_LST.xlsx
Raw_Billing.xlsx
Raw_RO_Attempts.xlsx
Silver_Output


In [3]:
import pandas as pd

file_path = "/lakehouse/default/Files/Raw_Billing.xlsx"

all_sheets = pd.read_excel(file_path, sheet_name=None, engine='openpyxl')

df = pd.concat(all_sheets.values(), ignore_index=True)




StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 5, Finished, Available, Finished, False)

In [4]:
df = df[df['Due Date (Print Doc)'] != 'Result']


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 6, Finished, Available, Finished, False)

In [5]:
# Optional: check shape


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 7, Finished, Available, Finished, False)

# Duplicate Removal

In [6]:
df = df.drop_duplicates(
    subset=[
        'Contract Number',
        'Calendar Year/Month',
        'Cycle Day',
        'Posting date',
        'BCM/Reversal Reason',
        'Due Date (Print Doc)',
        'Units Billed',
        'Net Amount'
    ],
    keep='first'
).reset_index(drop=True)


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 8, Finished, Available, Finished, False)

# Due Month Making


In [7]:
import pandas as pd
import numpy as np


def float_to_datetime(f):
    month_str, year_str = str(f).split(".")
    month = int(month_str)
    year = int(year_str)
    return pd.Timestamp(year=year, month=month, day=1)

# Apply function to the column
df['Month'] = df['Calendar Year/Month'].apply(float_to_datetime)

# Optional: formatted string for easy reading
df['Month'] = df['Month'].dt.strftime('%b-%Y')





StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 9, Finished, Available, Finished, False)

In [8]:
import pandas as pd
import numpy as np

# Convert Month string to datetime
df['Month_dt'] = pd.to_datetime(df['Month'], format='%b-%Y')

# Compute Due Month
df['Due Month'] = np.where(
    df['Cycle Day'].between(1, 10),             # Cycle Day 1-10 → same month
    df['Month_dt'],
    df['Month_dt'] + pd.DateOffset(months=1)   # Cycle Day 11-20 → next month
)

# Format as "Feb-2026"
df['Due Month'] = df['Due Month'].dt.strftime('%b-%Y')

# Optional: drop helper column
df.drop(columns=['Month_dt'], inplace=True)




StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 10, Finished, Available, Finished, False)

# Converting Date format


In [9]:
import pandas as pd


# Convert to datetime
df['Posting date'] = pd.to_datetime(df['Posting date'], format='%d.%m.%Y')




StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 11, Finished, Available, Finished, False)

# BCM Making

In [10]:
import pandas as pd

# Pivot table
pivot = pd.pivot_table(
    df,
    index="Contract Number",
    columns="BCM/Reversal Reason",
    values=["Units Billed", "Net Amount"],
    aggfunc="sum",
    fill_value=0
)

# 🔹 Add Total Columns BEFORE flattening
pivot[("Units Billed", "Total")] = pivot["Units Billed"].sum(axis=1)
pivot[("Net Amount", "Total")] = pivot["Net Amount"].sum(axis=1)

# 🔹 Flatten multi-level columns
pivot.columns = [f"{val}-{col}" for val, col in pivot.columns]

pivot = pivot.reset_index()




StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 12, Finished, Available, Finished, False)

In [11]:
import numpy as np

def define_bcm(row):

     # 6️⃣ Other group
    other_cols = [
        "Units Billed-CO", "Units Billed-KP", "Units Billed-RD",
        "Units Billed-RO", "Units Billed-RQ", "Units Billed-RS",
        "Units Billed-RU", "Units Billed-RY", "Units Billed-RZ"
    ]

    # 1️⃣ Reversal
    if row.get("Units Billed-Total", 0) < 0:
        return "Reversal"

    # 2️⃣ IRB
    elif row.get("Units Billed-BP", 0) > 0:
        return "IRB"

    elif row.get("Units Billed-MP", 0) > 0:
        return "IRB"

    # 3️⃣ Assessed
    elif row.get("Units Billed-X", 0) > 0:
        return "Assessed"

    # 4️⃣ Average
    elif row.get("Units Billed-A", 0) > 0:
        return "Average"

    # 5️⃣ Normal
    elif row.get("Units Billed-N", 0) > 0:
        return "Normal"

    elif row.get("Units Billed-C", 0) > 0:
        return "Normal"


    elif any(row.get(col, 0) > 0 for col in other_cols):
        return "IRB-Other"

    # 7️⃣ Fallback logic (all zero case)
    elif row.get("Units Billed-C", 0) == 0 and row.get("Units Billed-N", 0) == 0:
        return "Normal"

    elif row.get("Units Billed-A", 0) == 0:
        return "Average"

    elif row.get("Units Billed-X", 0) == 0:
        return "Assessed"

    elif row.get("Units Billed-MP", 0) == 0 or row.get("Units Billed-BP", 0) == 0:
        return "IRB-Other"

    return "Unknown"


pivot["Defined BCM"] = pivot.apply(define_bcm, axis=1)




StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 13, Finished, Available, Finished, False)

In [12]:
print(sorted(pivot["Defined BCM"].unique()))


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 14, Finished, Available, Finished, False)

['Assessed', 'Average', 'IRB', 'IRB-Other', 'Normal', 'Reversal']


# Outputs

In [13]:
!pip install XlsxWriter

StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 15, Finished, Available, Finished, False)

In [14]:


import pandas as pd

# Define output path in Lakehouse
file_path = "/lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx"

# Create Excel writer
with pd.ExcelWriter(file_path, engine="xlsxwriter") as writer:

    # Loop through each Due Month
    for due_month, data in df.groupby("Due Month"):

        # Excel sheet names max length = 31
        sheet_name = str(due_month)[:31]

        data.to_excel(writer, sheet_name=sheet_name, index=False)

print("File successfully saved to:", file_path)


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 16, Finished, Available, Finished, False)

File successfully saved to: /lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx


In [15]:
# Define output path in Lakehouse
file_path = "/lakehouse/default/Files/Silver_Output/Defined BCM.xlsx"

# Export pivot dataframe to Excel
pivot.to_excel(file_path, index=False)

print("Pivot exported successfully to:", file_path)


StatementMeta(, 861e6a46-b002-4be2-bdb2-95a11a505bb3, 17, Finished, Available, Finished, False)

Pivot exported successfully to: /lakehouse/default/Files/Silver_Output/Defined BCM.xlsx
